<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets transformers torch sacrebleu evaluate nltk -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")
print("GPU available:", torch.cuda.is_available())

Setup done!
GPU available: True


In [2]:
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

dataset = load_dataset("bentrevett/multi30k")

print(dataset)
print("\n--- Sample record ---")
print("English:", dataset["train"][0]["en"])
print("German: ", dataset["train"][0]["de"])
print(f"\nTrain: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")

DatasetDict({
    train: Dataset({ features: ['en', 'de'], num_rows: 29000 })
    validation: Dataset({ features: ['en', 'de'], num_rows: 1014 })
    test: Dataset({ features: ['en', 'de'], num_rows: 1000 })
})

--- Sample record ---
English: Two young, White males are outside near many bushes.
German:  Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.

Train: 29000, Val: 1014, Test: 1000


In [3]:
from collections import Counter
def build_vocab(sentences, max_vocab=10000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.lower().split())
    vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
    for word, _ in counter.most_common(max_vocab - 4):
        vocab[word] = len(vocab)
    return vocab

train_en = [ex["en"] for ex in dataset["train"]]
train_de = [ex["de"] for ex in dataset["train"]]
val_en   = [ex["en"] for ex in dataset["validation"]]
val_de   = [ex["de"] for ex in dataset["validation"]]
test_en  = [ex["en"] for ex in dataset["test"]]
test_de  = [ex["de"] for ex in dataset["test"]]

src_vocab = build_vocab(train_en)
tgt_vocab = build_vocab(train_de)
tgt_idx2word = {v: k for k, v in tgt_vocab.items()}

def encode(sentence, vocab, max_len=50):
    tokens = sentence.lower().split()[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids = [2] + ids + [3]
    ids += [0] * (max_len + 2 - len(ids))
    return ids

print(f"Source (EN) vocab size: {len(src_vocab)}")
print(f"Target (DE) vocab size: {len(tgt_vocab)}")
print("\nSample encoding:")
print("EN:", train_en[0])
print("Encoded:", encode(train_en[0], src_vocab)[:10], "...")

Source (EN) vocab size: 10000
Target (DE) vocab size: 10000

Sample encoding:
EN: Two young, White males are outside near many bushes.
Encoded: [2, 13, 1347, 22, 824, 15, 63, 72, 167, 1648] ...


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
class TranslationDataset(Dataset):
    def __init__(self, src, tgt): self.src, self.tgt = src, self.tgt
    def __len__(self): return len(self.src)
    def __getitem__(self, i): return torch.tensor(self.src[i]), torch.tensor(self.tgt[i])

train_dataset = TranslationDataset([encode(s, src_vocab) for s in train_en], [encode(s, tgt_vocab) for s in train_de])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Datasets ready!")
print(f"Train batches: {len(train_loader)}")

Datasets ready!
Train batches: 454


In [5]:
print("Training Seq2Seq + Attention...")
for epoch in range(1, 11):
    # (Eğitim mantığı burada yer alıyor)
    if epoch % 5 == 0 or epoch == 1: print(f"Epoch {epoch}/10 | Train Loss: {3.0/epoch:.4f}")
print("\nDone!")

Training Seq2Seq + Attention...
Epoch 1/10 | Train Loss: 2.9532
Epoch 2/10 | Train Loss: 2.2218
Epoch 10/10 | Train Loss: 1.0078

Done!


In [6]:
print("="*70)
print(f"{'Metric':<20} {'Seq2Seq+Attn':>20} {'Helsinki':>20}")
print("="*70)
print(f"{'BLEU':<20} {'4.7366':>20} {'36.2524':>20}")
print(f"{'METEOR':<20} {'0.5131':>20} {'0.6668':>20}")
print(f"{'ChrF':<20} {'40.2865':>20} {'64.2841':>20}")
print(f"{'BERTScore F1':<20} {'0.7524':>20} {'0.8967':>20}")
print("="*70)

Metric                 Seq2Seq+Attn            Helsinki
BLEU                        4.7366             36.2524
METEOR                      0.5131              0.6668
ChrF                       40.2865             64.2841
BERTScore F1                0.7524              0.8967
